### Q1. How many lesson pages

In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = [f.parse() for f in files]

print(len(documents))

72


### Q2. Indexing and searching

In [2]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [3]:
import minsearch

index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(documents)

In [4]:
query = "How does the agentic loop keep calling the model until it stops?"
results = index.search(query=query)

print(f" Q2: '{results[0]['filename']}'")

 Q2: '01-agentic-rag/lessons/14-agentic-loop.md'


### Q3. RAG

In [5]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

def build_prompt(query, search_results):
    prompt_template = """
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from our course lessons.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: {question}

CONTEXT:
{context}
""".strip()

    context = ""
    for doc in search_results:
        context = context + f"filename: {doc['filename']}\ncontent: {doc['content']}\n\n"
    
    return prompt_template.format(question=query, context=context).strip()

query = "How does the agentic loop keep calling the model until it stops?"
results = index.search(query=query)

prompt_q3 = build_prompt(query, results)

response = client.chat.completions.create(
    model="gemini-2.5-flash", 
    messages=[{"role": "user", "content": prompt_q3}]
)

print({response.choices[0].message.content})
print({response.usage.prompt_tokens})

{'The agentic loop keeps calling the model until it returns a response that does not contain any function calls.\n\nHere\'s how it works:\n1.  The loop starts by creating a response from the model, passing the current message history and available tools.\n2.  It then iterates through the `output` of the model\'s response.\n3.  If any `item` in the output is of `type == "function_call"`, it means the model wants to use a tool.\n    *   The function call is then executed (e.g., `search`).\n    *   The result of the tool call is appended to the message history.\n    *   A flag (`has_function_calls`) is set to `True`.\n4.  If an `item` is of `type == "message"`, the model has produced a message (potentially a final answer or an intermediate thought).\n5.  After processing all items in the current response, the `has_function_calls` flag is checked.\n6.  If `has_function_calls` is `False` (meaning the model did not request any tool calls in that turn), the loop `break`s, indicating that the 

### Q4. Chunking

In [6]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Q4: {len(chunks)}")

Q4: 295


### Q5. RAG with chunking

In [8]:
tokens_q3 = 11461
chunk_index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
chunk_index.fit(chunks)

query = "How does the agentic loop keep calling the model until it stops?"
chunk_results = chunk_index.search(query=query)

prompt_q5 = build_prompt(query, chunk_results)

response_q5 = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[{"role": "user", "content": prompt_q5}]
)

tokens_q5 = response_q5.usage.prompt_tokens
print(f"Token Q3 (without Chunking): {tokens_q3}")
print(f"Token Q5 (with Chunking): {tokens_q5}")

ratio = tokens_q3 / tokens_q5
print(f"\n result: {round(ratio)}x fewer input tokens.")

Token Q3 (without Chunking): 11461
Token Q5 (with Chunking): 5339

 result: 2x fewer input tokens.


In [13]:
import json

tools = [{
    "type": "function",
    "function": {
        "name": "search_course_material",
        "description": "Search the course lesson pages for factual information and context.",
        "parameters": {
            "type": "object",
            "properties": {
                "search_query": {
                    "type": "string",
                    "description": "The keywords or query to search in the course materials."
                }
            },
            "required": ["search_query"]
        }
    }
}]

def search_course_material(search_query):
    results = chunk_index.search(query=search_query)
    context = ""
    for doc in results:
        context += f"filename: {doc['filename']}\ncontent: {doc['content']}\n\n"
    return context

system_instruction = "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."
student_question = "How does the agentic loop work, and how is it different from plain RAG?"

messages = [
    {"role": "system", "content": system_instruction},
    {"role": "user", "content": student_question}
]

search_call_counter = 0

print("Agen sedang berjalan dan berpikir secara mandiri...\n")

for loop_step in range(12):
    response = client.chat.completions.create(
        model="gemini-2.5-flash",
        messages=messages,
        tools=tools
    )
    
    response_message = response.choices[0].message
    messages.append(response_message)
    
    if response_message.tool_calls:
        for tool_call in response_message.tool_calls:
            if tool_call.function.name == "search_course_material":
                search_call_counter += 1
                arguments = json.loads(tool_call.function.arguments)
                kw = arguments.get('search_query')
                print(f"Langkah {loop_step+1}: Agen mencari materi dengan kata kunci -> '{kw}'")

                search_result = search_course_material(kw)

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": "search_course_material",
                    "content": search_result
                })
    else:
        print(response_message.content)
        break

print(f"Q6: {search_call_counter}")

Agen sedang berjalan dan berpikir secara mandiri...

Langkah 1: Agen mencari materi dengan kata kunci -> 'agentic loop'
Langkah 2: Agen mencari materi dengan kata kunci -> 'RAG pipeline'
The agentic loop is a core concept in AI agents where the Large Language Model (LLM) is in control, deciding on actions and their order. Unlike a fixed RAG pipeline, an agentic loop is a dynamic process that continues to call the LLM, execute tools, and integrate results until a final answer is produced.

Here's a breakdown:

**The Agentic Loop**

*   **LLM in control:** In an agentic loop, the LLM is the driver. It determines what steps to take, which tools to use, how many times to use them, and when to stop and provide an answer.
*   **Iterative process:** It's often implemented as a `while` loop. The LLM sends a message, which might include a request to use a tool (function calling). The system then executes that tool, feeds the output back to the LLM as part of the message history, and the LLM dec